In [1]:
# Basic libraries
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [2]:
# Define project paths
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load processed renewal dataset
df = pd.read_csv(
    PROCESSED_DIR / "renewal_model_data.csv"
)

print("Shape:", df.shape)

df.head()

Shape: (19204, 44)


,RENEWAL_ID,POLICY_ID,CUSTOMER_ID,RENEWAL_DUE_DATE,OLD_PREMIUM,NEW_PREMIUM,PREMIUM_INCREASE_PCT,CLAIM_HISTORY_FLAG,LOYALTY_YEARS,RENEWAL_PROBABILITY,...,AGE,GENDER,MARITAL_STATUS,OCCUPATION,ANNUAL_INCOME,STATE,CREDIT_SCORE,CUSTOMER_RISK_SEGMENT,TARGET_RENEWED,TARGET_CHURN
0,REN00004654,POL000004683,CUST00068704,2025-01-01,19919,20853,4.688990,No,6,0.9280,...,38,Male,Married,Self Employed,293916,Delhi,812,Medium,1,0
1,REN00009235,POL000028226,CUST00000909,2025-01-01,2303,2631,14.242293,No,1,0.7940,...,49,Male,Married,Professional,322439,Kerala,693,Low,1,0
2,REN00014001,POL000083919,CUST00034993,2025-01-01,17016,18877,10.936765,Yes,6,0.8039,...,26,Male,Married,Salaried,206728,Gujarat,764,Medium,1,0
3,REN00013440,POL000078165,CUST00024696,2025-01-01,32198,35317,9.686937,No,7,0.9107,...,52,Other,Married,Salaried,1366260,Kerala,683,Low,1,0
4,REN00012005,POL000059929,CUST00027203,2025-01-01,10820,12384,14.454713,No,6,0.8311,...,18,Female,Single,Self Employed,264864,Tamil Nadu,667,Low,1,0


In [3]:
# Check renewal outcome distribution
df["RENEWAL_STATUS"].value_counts()

RENEWAL_STATUS
Renewed      17334
Lapsed        1463
Cancelled      407
Name: count, dtype: int64

In [4]:
# Remove records where final outcome is still pending
df = df[
    df["RENEWAL_STATUS"] != "Pending"
].copy()

# Business-aligned target:
# 1 = Non-renewal/churn
# 0 = Renewed
df["TARGET_CHURN"] = np.where(
    df["RENEWAL_STATUS"] == "Renewed",
    0,
    1
)

df["TARGET_CHURN"].value_counts()

TARGET_CHURN
0    17334
1     1870
Name: count, dtype: int64

In [8]:
# Numerical input features
numeric_features = [
    "AGE",
    "ANNUAL_INCOME",
    "CREDIT_SCORE",
    "SUM_INSURED",
    "ANNUAL_PREMIUM",
    "RISK_SCORE",
    "PREMIUM_INCREASE_PCT",
    "CUSTOMER_TENURE_YEARS",
    "TOTAL_PAYMENTS",
    "AVG_PAYMENT_DELAY",
    "MAX_PAYMENT_DELAY",
    "TOTAL_CLAIMS",
    "TOTAL_CLAIM_AMOUNT",
    "AVG_CLAIM_AMOUNT",
    "MAX_CLAIM_AMOUNT"
]

# Categorical input features
categorical_features = [
    "GENDER",
    "MARITAL_STATUS",
    "OCCUPATION",
    "STATE",
    "CUSTOMER_RISK_SEGMENT",
    "POLICY_TYPE",
    "SALES_CHANNEL",
    "PAYMENT_MODE",
    "RISK_BAND"
]

# Combine all model features
all_features = numeric_features + categorical_features

In [9]:
# Validate that every required feature exists
missing_features = [
    col for col in all_features
    if col not in df.columns
]

print("Missing features:", missing_features)

Missing features: []


In [10]:
# X contains model inputs
X = df[all_features].copy()

# y contains target
y = df["TARGET_CHURN"].copy()

print("X:", X.shape)
print("y:", y.shape)

X: (19204, 24)
y: (19204,)
